# Rainbow (IQN) with ALE/Seaquest-v5

This notebook loads [config_rainbow.yaml](config_rainbow.yaml) and trains the
`RainbowDQNAgent` on `ALE/Seaquest-v5` — the **sample-efficiency benchmark** for the
low-rank Hankel study. It is the Rainbow counterpart to the DQN baseline in
[exp1.ipynb](exp1.ipynb): **same env, same Atari preprocessing, same training
schedule**, so the two are directly comparable. The only thing that changes is the
agent (and therefore the network wiring).

Rainbow here = Double DQN + Dueling + Prioritised replay + Multi-step returns +
Distributional (**IQN**, in place of C51) + Noisy Nets. See
[src/agents/README.md](../../src/agents/README.md) for the full manual.

**Key wiring difference from the baseline.** The DQN notebook passes a whole
`DuelingNatureCNN` (trunk + heads). Rainbow instead takes only an **encoder** — the
Nature conv trunk producing a `(B, 3136)` feature vector — and builds the dueling +
noisy + IQN head *inside* the agent. So `nn_extra_kwargs` carries encoder args only;
the agent infers `feature_dim` and `n_actions` itself.

## Imports

In [2]:
import sys, pathlib
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import ale_py
import gymnasium as gym
gym.register_envs(ale_py)

SRC = pathlib.Path.cwd().parents[1] / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Config-driven wiring: change config_rainbow.yaml and it flows through these
# builders without editing the notebook. Hankel analysis is driven by
# analysis.hankel_sweep in the config (dispatched inside the training loop).
from experiment import load_config, build_env, build_agent, train, make_run_logger
from agents.rainbow_agent import RainbowDQNAgent
from analysis.registry import resolve_methods
from analysis.low_rank.rank import row_rank_property_check
from analysis.visualisations.heatmaps import plot_matrix_heatmap

## Reading the config file

Note the non-default filename — this experiment has two configs (`config.yaml` for the DQN baseline, `config_rainbow.yaml` here).

In [3]:
cfg = load_config("config_rainbow.yaml")   # loads yaml, resolves device, seeds torch/numpy
print("device:", cfg["experiment"]["_device"])
cfg

device: cuda


{'experiment': {'name': 'dqn_seaquest_rainbow',
  'seed': 52,
  'device': 'cuda',
  'save_artifacts': True,
  '_device': 'cuda'},
 'environment': {'name': 'ALE/Seaquest-v5',
  'render_mode': None,
  'discrete_config': None,
  'normalise': {'action': {}, 'state': {}},
  'clip': {'action': False, 'state': {}},
  'atari': {'noop_max': 30,
   'frame_skip': 4,
   'screen_size': 84,
   'terminal_on_life_loss': True,
   'grayscale_obs': True,
   'grayscale_newaxis': False,
   'scale_obs': False,
   'frame_stack': 4},
  'reward': {'scale': 0.01}},
 'network': None,
 'agent': {'replay_buffer_capacity': 250000,
  'batch_size': 128,
  'nn_learning_rate': 0.0001,
  'discount_factor': 0.99,
  'n_step': 3,
  'n_quantiles': 32,
  'n_quantiles_target': 32,
  'n_quantiles_act': 32,
  'n_cos': 64,
  'head_hidden': 512,
  'huber_kappa': 1.0,
  'dueling': True,
  'noisy_sigma0': 1.0,
  'double': True,
  'per_alpha': 0.5,
  'per_beta_start': 0.4,
  'per_beta_increment': 2e-06,
  'per_eps': 1e-05,
  'TD_LR'

## Creating the Environment

The `atari` block triggers the Atari branch of `make_environment`, wiring
`AtariPreprocessing` followed by `FrameStackObservation`. Resulting
`observation_space` is `Box(0, 255, (4, 84, 84), uint8)` and `action_space` is
`Discrete(18)` — identical to the DQN baseline.

In [4]:
env = build_env(cfg)
obs_shape = env.observation_space.shape   # (4, 84, 84)
n_actions = env.action_space.n
print("obs_shape:", obs_shape, "dtype:", env.observation_space.dtype, "n_actions:", n_actions)

obs_shape: (4, 84, 84) dtype: uint8 n_actions: 18


A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]


## Encoder (Nature conv trunk only)

For Rainbow the externally-supplied network is just the **encoder** — the Nature DQN
convolutional trunk, with the Q-head removed. It maps a `(4, 84, 84)` uint8 frame
stack to a `(B, 3136)` feature vector and **owns its own normalisation** (`x/255`),
exactly like the baseline's trunk.

Everything Rainbow-specific — the cosine-τ IQN embedding, the dueling value/advantage
streams, and the `NoisyLinear` layers — is built by `RainbowDQNAgent` around this
encoder (see [src/agents/rainbow_agent.py](../../src/agents/rainbow_agent.py)). That
is why there are no `n_actions` / `fc_hidden` here: the agent derives `feature_dim`
from a dummy forward and reads `n_actions` from the env, and the head width is
`agent.head_hidden` in the config.

| Layer | Spec | Output |
|---|---|---|
| Conv1 | 4 → 32, kernel 8, stride 4 | 32×20×20 |
| Conv2 | 32 → 64, kernel 4, stride 2 | 64×9×9 |
| Conv3 | 64 → 64, kernel 3, stride 1 | 64×7×7 → flatten 3136 |

In [5]:
class NatureEncoder(nn.Module):
    """Nature DQN conv trunk as a plain feature extractor (no head).
    Maps a (C, 84, 84) uint8 frame stack to features of shape (B, 3136).
    Owns its own normalisation, so RainbowDQNAgent can stay input-agnostic."""
    def __init__(self, in_channels):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),          nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),          nn.ReLU(),
            nn.Flatten(),
        )

    def forward(self, x):
        return self.features(x.float() / 255.0)

## Creating the Agent

`build_agent` takes an optional `agent_cls` (default `QAgent`); pass
`RainbowDQNAgent` to build the benchmark. `nn_extra_kwargs` carries **encoder args
only** — no `n_actions`, no `fc_hidden`. Every Rainbow hyperparameter comes from
`cfg["agent"]` (n-step, IQN quantiles, PER, noisy σ₀, …).

In [6]:
# The encoder class + its construction args are genuine code (not config keys), so
# they stay explicit; every agent hyperparameter comes from cfg["agent"].
nn_extra_kwargs = {"in_channels": obs_shape[0]}   # encoder args only
agent = build_agent(cfg, env, NatureEncoder, nn_extra_kwargs, agent_cls=RainbowDQNAgent)

## Resume from a previous run (optional)

Set `RESUME_RUN` to a `runs/<timestamp>` directory to continue training from its
`checkpoints/latest.pt`; set it to `None` to train from scratch. What the checkpoint
carries: policy + target weights and optimiser state. What it does **not** carry: the
replay buffer (starts empty, hence the small `warmup_steps` refill from the loaded —
already competent — policy) and the PER β anneal (restarts at `per_beta_start`; minor).
`no_episodes` is reduced by the episodes the previous run completed (counted from its
`rewards.csv`), so the total across runs stays the configured budget.

Run this cell **once**, right after building the agent — re-running it later in the
session would reload the old weights over your continued training. The logger cell
below then creates a fresh `runs/<timestamp>/` for the continuation, leaving the
original run's artifacts untouched; concatenate the two `rewards.csv` files when
plotting the full curve.

If you only interrupted the kernel (agent still in memory), skip this cell — set
`cfg["training"]["warmup_steps"] = 0`, re-run the logger cell, and train.

In [7]:
# Resume from a previous run's latest checkpoint (None = fresh training).
RESUME_RUN = "runs/20260713-155148"

if RESUME_RUN is not None:
    import csv
    ckpt = pathlib.Path(RESUME_RUN) / "checkpoints" / "latest.pt"
    agent.load(ckpt)
    with open(pathlib.Path(RESUME_RUN) / "rewards.csv") as f:
        episodes_done = len(list(csv.reader(f))) - 1          # minus header row
    cfg["training"]["no_episodes"] = max(cfg["training"]["no_episodes"] - episodes_done, 0)
    cfg["training"]["warmup_steps"] = 5000    # small refill of the empty replay buffer
    print(f"resumed weights + optimiser from {ckpt}")
    print(f"episodes already done: {episodes_done} -> continuing for "
          f"{cfg['training']['no_episodes']} (warmup {cfg['training']['warmup_steps']})")

resumed weights + optimiser from runs/20260713-155148/checkpoints/latest.pt
episodes already done: 3201 -> continuing for 11799 (warmup 5000)


## Analysis (Low Rank)

The focus of this benchmark is the **Hankel low-rank sweep**, configured by the
`analysis.hankel_sweep` block in [config_rainbow.yaml](config_rainbow.yaml) and
dispatched inside the training loop every `ep_freq` episodes. It rolls out the
current policy over several seeds (`n_rollouts`, seed = `base_seed + r`) and builds
Hankel matrices of the V, Q and A sequences. Because Rainbow is genuinely **dueling**,
`value_advantage` exposes learned V(s) and mean-centred A(s,·) streams — so the
Hankel **A** trace is a real signal here, not the trivially rank-deficient
`Q − max Q` of a vanilla head.

With `sub_trajectory.enabled`, each rollout is swept over growing prefixes `0→τ`, and
`n_figures` spectra (rollout 0) are rendered across the sweep. **`save_heatmaps:
true`** additionally saves a heatmap of each rendered Hankel matrix next to its
spectrum, so `figures/` carries both views at every tick. Per-rollout metrics land in
`hankel_sweep.csv`; raw Q/V/A sequences go to `trajectories/*.npz`.

The discretised Q-matrix analysis from the baseline is intentionally dropped
(`post_methods: []`) — this run is about the Hankel structure.

**Run artifacts.** A `RunLogger` snapshots everything under `runs/<timestamp>/`
(gitignored): a frozen copy of `config_rainbow.yaml`, `rewards.csv`,
`hankel_sweep.csv`, `trajectories/`, the spectra + heatmaps as
`figures/epNNNNNN_*.png`, and checkpoints (`latest`/`best`/`final`). Restore any with
`agent.load(path)`.

In [8]:
# All artifacts land under runs/<timestamp>/ when experiment.save_artifacts is set;
# otherwise logger is None and analysis renders inline. We pass the non-default
# config filename so the frozen snapshot is config_rainbow.yaml (not config.yaml).
logger = make_run_logger(cfg, config_path="config_rainbow.yaml")
if logger:
    print("run artifacts ->", logger.dir)

run artifacts -> /home/souparna/Low-Rank-RL-TL/experiments/dqn_seaquest/runs/20260713-230612


## Agent Training

Same schedule as the DQN baseline. With noisy-net exploration (no ε schedule),
prioritised replay and multi-step distributional targets, Rainbow is expected to be
markedly more sample-efficient than the baseline on Seaquest's long oxygen-loop credit
assignment — that gap is exactly what this benchmark measures.

In [9]:
rewards = train(cfg, agent, env, run_logger=logger)

  0%|          | 0/11799 [00:00<?, ?it/s]

episode 0 avg_rewarg: 460.0


  0%|          | 21/11799 [04:03<22:12:36,  6.79s/it] 

episode 20 avg_rewarg: 369.0


  0%|          | 41/11799 [05:57<19:15:12,  5.89s/it]

...

episode 4500 avg_rewarg: 866.0


 38%|███▊      | 4521/11799 [9:21:16<11:03:51,  5.47s/it]

episode 4520 avg_rewarg: 841.0


 38%|███▊      | 4539/11799 [9:23:19<15:01:01,  7.45s/it]


KeyboardInterrupt: 

## Training and Analysis Plots

In [ ]:
rewards = np.asarray(rewards, dtype=float)
plt.figure(figsize=(8, 4))
plt.plot(rewards, alpha=0.35, label="episode reward")
if len(rewards) >= 10:
    k = 10
    ma = np.convolve(rewards, np.ones(k) / k, mode="valid")
    plt.plot(range(k - 1, len(rewards)), ma, label=f"{k}-ep moving avg")
plt.xlabel("episode"); plt.ylabel("total reward"); plt.title("Rainbow (IQN) on ALE/Seaquest-v5")
plt.legend()
if logger:
    plt.savefig(logger.dir / "reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## Post-training analysis

The Hankel sweep already ran every `ep_freq` during training (`hankel_sweep.csv` /
`figures/`). `post_methods` is empty for this benchmark, so this cell is a no-op
placeholder kept for structural parity with the baseline notebook — add methods to
`analysis.post_methods` in the config to populate it.

In [ ]:
methods = resolve_methods(cfg["analysis"].get("methods", []) + cfg["analysis"].get("post_methods", []))
for method, names in methods:
    results = method(agent=agent, env=env)
    if not isinstance(results, tuple):
        results = (results,)
    for matrix, name in zip(results, names):
        print(name)
        plot_matrix_heatmap(matrix, name, save_to=logger.figure_path(f"{name} heatmap") if logger else None)
        r, sr, spk, shape, irs, ics, rc, cc, nzr, nzc = row_rank_property_check(matrix, name, save_to=logger.figure_path(name) if logger else None)
        print(f"eff_rank: {r}, stable_rank: {sr:.2f}, spikiness: {spk:.2f}, shape: {shape}, non-zero rows :{nzr}, non-zero cols:{nzc}")

## Greedy rollout video

Record one greedy (`act_greedy`) episode of the trained agent and display it inline.
Note the greedy action still uses the noisy weights (exploration is baked into the
net); pass a deterministic eval path if you need noise-free rollouts.

In [ ]:
from analysis.visualisations.rollout_video import record_greedy_episode
from IPython.display import Video
import glob

video_dir = "videos"
eval_env = build_env(cfg, render_mode="rgb_array")
prefix = record_greedy_episode(agent, eval_env, video_dir, episode=0, seed=cfg["experiment"]["seed"])
mp4 = sorted(glob.glob(f"{video_dir}/{prefix}-*.mp4"))[-1]
print("saved:", mp4)
Video(mp4, embed=True)